In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
%matplotlib inline

In [ ]:
words = open('names.txt', 'r').read().splitlines()

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [ ]:
VOCAB_SIZE   = 27
CONTEXT_SIZE = 3
EMBED_DIM    = 10
N_HIDDEN     = 200

In [ ]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * CONTEXT_SIZE
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr,  Ytr  = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte,  Yte  = build_dataset(words[n2:])

#MLP: E01

increased block_size to 6, embed_dim to 12, hidden to 300.
used kaiming init on W1 and scaled down W2 to 0.01 so the starting loss is near uniform.
lr decay: 0.1 for 150k steps then 0.01. ran 200k total steps with batch size 128.
got a dev loss of 2.07, beating andrej's 2.2.


In [ ]:
#MLP: E02

'''
we have 27 characters (26 alphabets + '.').
if all probabilities are equal (1/27), the cross entropy loss = -log(1/27)
'''
nll = -math.log(1/27)
print(nll)

#after init with random weights, the logits are all over the place so the loss is huge (~25)
#to get a starting loss close to the uniform value, we scale W2 down to 0.01 and zero b2
#this makes the initial logits near zero -> softmax is flat -> loss is near -log(1/27)
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)
b = torch.randn(27,       generator=g, requires_grad=True)
#naive loss is around 25. multiplying W2 by 0.01 brings it down to ~3.29

#MLP: E03

idea from bengio 2003: add a direct connection from the embedding layer to the output,
skipping the hidden layer. so logits = h @ W2 + x @ W3 + b2.
the W3 term is a linear shortcut. this helps when the hidden layer is a bottleneck.
trained for 30k steps, got dev loss of 2.27.
marginal improvement over baseline. makes more of a difference with a smaller hidden layer.


In [ ]:
#PART 2: E01

#all weights and biases initialised to zero. W=0, b=0, so every neuron outputs zero.
#for gradients:
#d(loss)/dW = x * (upstream grad)  ->  since x (or emb) = 0, dL/dW = 0. weights never move.
#d(loss)/db = 1 * (upstream grad)  ->  local gradient is 1 so biases do get updated.
#so b2 trains (adjusts logit biases, learns unigram frequencies)
#but W1, W2, C stay at zero the whole time.
#network gets stuck around loss 2.8 because it can only express a unigram model via b2.

In [ ]:
#verifying with actual gradients
Cz  = torch.zeros((VOCAB_SIZE, EMBED_DIM))
W1z = torch.zeros((CONTEXT_SIZE * EMBED_DIM, N_HIDDEN))
b1z = torch.zeros(N_HIDDEN)
W2z = torch.zeros((N_HIDDEN, VOCAB_SIZE))
b2z = torch.zeros(VOCAB_SIZE)
for p in [Cz, W1z, b1z, W2z, b2z]: p.requires_grad = True

emb    = Cz[Xtr[:32]].view(-1, CONTEXT_SIZE * EMBED_DIM)
h      = torch.tanh(emb @ W1z + b1z)
logits = h @ W2z + b2z
loss   = F.cross_entropy(logits, Ytr[:32])
loss.backward()

print(f'b2  grad: {b2z.grad.abs().max():.4f}  <- nonzero, trains')
print(f'W2  grad: {W2z.grad.abs().max():.4f}  <- zero, dead')
print(f'W1  grad: {W1z.grad.abs().max():.4f}  <- zero, dead')
print(f'C   grad: {Cz.grad.abs().max():.4f}   <- zero, dead')

In [ ]:
#PART 2: E02
#INTUITION:
#LINEAR LAYER: y = xW              (no bias, standard before BN)
#BATCHNORM:    z = gamma*(y-mu)/sigma + beta
#
#substituting y:
#z = x * W*(gamma/sigma)  +  (beta - gamma*mu/sigma)
#
#folding:
#W_new = W * (gamma / sigma)
#b_new = beta - gamma * mu / sigma
#
#so the BN layer disappears entirely at inference - same result, fewer ops

In [ ]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True, generator=None):
        self.W = torch.randn(fan_in, fan_out, generator=generator) * fan_in**-0.5
        self.b = torch.zeros(fan_out) if bias else None
    def __call__(self, x):
        self.out = x @ self.W + (self.b if self.b is not None else 0)
        return self.out
    def parameters(self): return [self.W] + ([self.b] if self.b is not None else [])

class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps=eps; self.momentum=momentum; self.training=True
        self.gamma=torch.ones(dim); self.beta=torch.zeros(dim)
        self.running_mean=torch.zeros(dim); self.running_var=torch.ones(dim)
    def __call__(self, x):
        if self.training:
            xmean=x.mean(0,keepdim=True); xvar=x.var(0,keepdim=True,unbiased=True)
            with torch.no_grad():
                self.running_mean=(1-self.momentum)*self.running_mean+self.momentum*xmean
                self.running_var =(1-self.momentum)*self.running_var +self.momentum*xvar
        else: xmean,xvar=self.running_mean,self.running_var
        xhat=(x-xmean)/torch.sqrt(xvar+self.eps)
        self.out=self.gamma*xhat+self.beta; return self.out
    def parameters(self): return [self.gamma, self.beta]

class Tanh:
    def __call__(self, x): self.out=torch.tanh(x); return self.out
    def parameters(self): return []

In [ ]:
g = torch.Generator().manual_seed(2147483647)
Cemb = torch.randn((VOCAB_SIZE, EMBED_DIM), generator=g) * 0.1
l1  = Linear(CONTEXT_SIZE*EMBED_DIM, 64, bias=False, generator=g)
bn1 = BatchNorm1d(64)
l2  = Linear(64, 64, bias=False, generator=g)
bn2 = BatchNorm1d(64)
l3  = Linear(64, VOCAB_SIZE, generator=g)

layers = [l1, bn1, Tanh(), l2, bn2, Tanh(), l3]
params = [Cemb] + [p for lay in layers for p in lay.parameters()]
for p in params: p.requires_grad = True

for i in range(10000):
    ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)
    x  = Cemb[Xtr[ix]].view(-1, CONTEXT_SIZE*EMBED_DIM)
    for lay in layers: x = lay(x)
    loss = F.cross_entropy(x, Ytr[ix])
    for p in params: p.grad = None
    loss.backward()
    lr = 0.1 if i < 8000 else 0.01
    for p in params: p.data -= lr * p.grad

for lay in layers:
    if isinstance(lay, BatchNorm1d): lay.training = False

In [ ]:
#fold BN into linear and verify
def fold_bn(linear, bn):
    W     = linear.W.detach()
    sigma = torch.sqrt(bn.running_var.detach() + bn.eps)
    W_new = W * (bn.gamma.detach() / sigma)
    b_new = bn.beta.detach() - bn.gamma.detach() * bn.running_mean.detach() / sigma
    return W_new, b_new

W1_new, b1_new = fold_bn(l1, bn1)
W2_new, b2_new = fold_bn(l2, bn2)

s = Cemb[Xdev[:32]].view(-1, CONTEXT_SIZE*EMBED_DIM)
with torch.no_grad():
    # original (with BN)
    xo = s
    for lay in layers: xo = lay(xo)
    # folded (no BN)
    xf = torch.tanh(s  @ W1_new + b1_new)
    xf = torch.tanh(xf @ W2_new + b2_new)
    xf = xf @ l3.W + l3.b

print(f'max diff: {(xo - xf).abs().max().item():.2e}')  #should be near zero